In [1]:
# ============================================================
# M5.1 — HEAT-KERNEL/BESSEL FINITE CERTIFICATE + TAIL PIN TEST
# ============================================================
#
# Goal:
#   Test the scalar ceiling reduction
#
#       B_L := alpha_W(L)^2 T_C(L)
#              - (3/(32*pi^2))*log(L/4)
#              < 57/1210
#
#   for integer L >= 4 over a large finite scan.
#
# If the remaining analytic theta-tail lemma proves the same inequality
# for all L beyond the scan, then:
#
#       T_C < 1/8
#       N*_C >= 8
#       T_full < 9/64
#       N* >= 7
#
# This script uses the exact Bessel/heat-kernel representation:
#
#   Y_L = alpha_W^2 T_C
#       = (3/4) ∫_0^∞ t exp(-mu_L^2 t)
#               (Phi_L(t)^4 - L^-4) dt
#
# where
#
#   mu_L^2 = m0^2 / alpha_W = 48 / (beta(L) L^2)
#   Phi_L(t) = sum_{j in Z} I_{jL}(2t) exp(-2t)
#
# and beta(L)=5.6 + gamma log(L/4), gamma=11/(8*pi^2).
#
# ============================================================

import math
import time
import numpy as np

try:
    from scipy.special import ive
    from scipy.integrate import quad
except Exception as e:
    raise RuntimeError(
        "SciPy is required for this block. In Colab, run: !pip install scipy"
    ) from e

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

BETA0 = 5.6
GAMMA = 11.0 / (8.0 * math.pi**2)
A_LOG = 3.0 / (32.0 * math.pi**2)
B_CRIT = 57.0 / 1210.0

# Default finite scan.
# 4096 is usually quick in Colab. Increase only after first run.
L_SCAN_MAX = 4096

# Numerical tolerances for quadrature/Bessel truncation.
RTOL = 2e-11
ATOL = 2e-13
BESSEL_REL_CUTOFF = 1e-16

print("=" * 100)
print("M5.1 HEAT-KERNEL/BESSEL FINITE CERTIFICATE")
print("=" * 100)
print(f"L_SCAN_MAX        = {L_SCAN_MAX}")
print(f"gamma             = {GAMMA:.18f}")
print(f"A_log             = {A_LOG:.18f}")
print(f"B_crit = 57/1210  = {B_CRIT:.18f}")
print(f"threshold T_C     = 1/8 = {1/8:.18f}")
print()

# ------------------------------------------------------------
# AF diagonal
# ------------------------------------------------------------

def beta_of_L(L: int) -> float:
    return BETA0 + GAMMA * math.log(L / 4.0)

def mu2_of_L(L: int) -> float:
    # m0^2 = 8/L^2 on the diagonal, alpha_W = beta/6.
    # mu^2 = m0^2 / alpha_W = (8/L^2) / (beta/6)
    return 48.0 / (beta_of_L(L) * L * L)

# ------------------------------------------------------------
# Exact heat-kernel/Bessel Phi_L
# ------------------------------------------------------------

def Phi_L_bessel(L: int, t: float) -> float:
    """
    Exact heat-kernel identity:

        Phi_L(t) = exp(-2t) * sum_{j in Z} I_{jL}(2t)
                 = sum_{j in Z} ive(jL, 2t)

    scipy.special.ive(v,z) = exp(-abs(z)) I_v(z), so for z=2t:
        ive(jL,2t)=exp(-2t)I_{jL}(2t).

    The j=0 term is included once, j>=1 terms twice.
    """
    out = float(ive(0, 2.0 * t))

    j = 1
    while True:
        term = 2.0 * float(ive(j * L, 2.0 * t))
        out_new = out + term

        if term <= BESSEL_REL_CUTOFF * max(abs(out_new), 1e-300):
            return out_new

        out = out_new
        j += 1

        if j > 200000:
            raise RuntimeError(
                f"Bessel image sum did not truncate: L={L}, t={t}, partial={out}"
            )

def Y_bessel(L: int):
    """
    Computes Y_L = alpha_W^2 T_C using the exact heat-kernel representation.

    Returns:
        Y_upper, quad_abs_err, tail_bound, cutoff_T
    """
    mu2 = mu2_of_L(L)
    Ns_inv = 1.0 / (float(L) ** 4)

    # Slowest nonzero mode of the lattice Laplacian.
    lam_min = 4.0 * math.sin(math.pi / L) ** 2

    # Very conservative finite cutoff. The tail is exponentially small because
    # Phi_L^4 - L^-4 decays at least on the nonzero spectral gap scale.
    T_cut = 80.0 / (mu2 + lam_min)

    def integrand(t):
        p = Phi_L_bessel(L, t)
        diff = p**4 - Ns_inv
        if diff < 0 and abs(diff) < 1e-18:
            diff = 0.0
        return t * math.exp(-mu2 * t) * diff

    val, err = quad(
        integrand,
        0.0,
        T_cut,
        epsabs=ATOL,
        epsrel=RTOL,
        limit=400
    )

    # Conservative positive tail envelope:
    # Phi_L^4 - L^-4 is bounded by a nonzero-gap exponential envelope.
    # This is intentionally loose; for the scanned range it is tiny.
    kappa = mu2 + lam_min
    tail = (T_cut + 1.0 / kappa) * math.exp(-kappa * T_cut) / kappa

    Y_upper = 0.75 * (val + abs(err) + tail)
    return Y_upper, abs(err), tail, T_cut

# ------------------------------------------------------------
# Optional dense closed-form cross-check for selected L
# ------------------------------------------------------------

def Y_dense_sum(L: int, chunk_elems: int = 4_000_000) -> float:
    """
    Direct closed-form finite lattice sum, independent of quadrature:

        Y_L = (3/(4L^4)) sum_{n != 0} (mu^2 + w(n))^-2

    Uses reduced 1D multiplicities.
    """
    mu2 = mu2_of_L(L)

    ns = np.arange(L // 2 + 1)
    vals = 4.0 * np.sin(np.pi * ns / L) ** 2

    mult = np.full(len(vals), 2.0)
    mult[0] = 1.0
    if L % 2 == 0:
        mult[-1] = 1.0

    V2 = (vals[:, None] + vals[None, :]).ravel()
    C2 = (mult[:, None] * mult[None, :]).ravel()

    nB = len(V2)
    step = max(1, chunk_elems // nB)

    S2 = 0.0
    for i in range(0, nB, step):
        R = 1.0 / (mu2 + (V2[i:i + step, None] + V2[None, :]))
        W = C2[i:i + step, None] * C2[None, :]
        S2 += float(np.sum(W * R * R))

    # Remove zero mode.
    S2 -= 1.0 / (mu2 * mu2)

    return 3.0 * S2 / (4.0 * float(L) ** 4)

# ------------------------------------------------------------
# Scan
# ------------------------------------------------------------

def row_for_L(L: int):
    Y, qerr, tail, T_cut = Y_bessel(L)
    x = math.log(L / 4.0)
    B_L = Y - A_LOG * x

    beta = beta_of_L(L)
    T_C = 36.0 * Y / (beta * beta)

    return {
        "L": L,
        "beta": beta,
        "mu2": mu2_of_L(L),
        "Y": Y,
        "B_L": B_L,
        "B_margin": B_CRIT - B_L,
        "T_C": T_C,
        "Nstar_C_floor": int(1.0 // T_C),
        "quad_err": qerr,
        "tail": tail,
        "T_cut": T_cut,
    }

start = time.time()

rows = []
max_B_row = None
max_TC_row = None
min_margin_row = None

print("Scanning integer L...")
for L in range(4, L_SCAN_MAX + 1):
    r = row_for_L(L)
    rows.append(r)

    if max_B_row is None or r["B_L"] > max_B_row["B_L"]:
        max_B_row = r
    if max_TC_row is None or r["T_C"] > max_TC_row["T_C"]:
        max_TC_row = r
    if min_margin_row is None or r["B_margin"] < min_margin_row["B_margin"]:
        min_margin_row = r

    if L in [4, 5, 6, 8, 12, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096] or L % 512 == 0:
        print(
            f"L={L:5d}  "
            f"Y={r['Y']:.12f}  "
            f"B_L={r['B_L']:.12f}  "
            f"margin={r['B_margin']:.12f}  "
            f"T_C={r['T_C']:.12f}  "
            f"N*_C={r['Nstar_C_floor']:3d}"
        )

elapsed = time.time() - start

print()
print("=" * 100)
print("FINITE SCAN SUMMARY")
print("=" * 100)
print(f"elapsed seconds       = {elapsed:.2f}")
print(f"scanned L             = 4 ... {L_SCAN_MAX}")
print()
print("max B_L row:")
print(json_row := {
    "L": max_B_row["L"],
    "beta": max_B_row["beta"],
    "Y": max_B_row["Y"],
    "B_L": max_B_row["B_L"],
    "Bcrit_margin": max_B_row["B_margin"],
    "T_C": max_B_row["T_C"],
    "Nstar_C_floor": max_B_row["Nstar_C_floor"],
})
print()
print("max T_C row in finite scan:")
print({
    "L": max_TC_row["L"],
    "beta": max_TC_row["beta"],
    "Y": max_TC_row["Y"],
    "B_L": max_TC_row["B_L"],
    "Bcrit_margin": max_TC_row["B_margin"],
    "T_C": max_TC_row["T_C"],
    "Nstar_C_floor": max_TC_row["Nstar_C_floor"],
})
print()

finite_pass = all(r["B_L"] < B_CRIT for r in rows)
tc_pass = all(r["T_C"] < 1.0 / 8.0 for r in rows)

print(f"finite B_L < Bcrit pass?  {finite_pass}")
print(f"finite T_C < 1/8 pass?    {tc_pass}")
print()

# ------------------------------------------------------------
# Dense cross-checks against the quadrature method
# ------------------------------------------------------------

print("=" * 100)
print("DENSE CLOSED-FORM CROSS-CHECKS")
print("=" * 100)

DENSE_CHECK_LS = [4, 8, 16, 32, 64, 128, 256]
for L in DENSE_CHECK_LS:
    if L > L_SCAN_MAX:
        continue
    t0 = time.time()
    Yd = Y_dense_sum(L)
    rb = rows[L - 4]
    diff = abs(Yd - rb["Y"])
    rel = diff / max(abs(Yd), 1e-300)
    print(
        f"L={L:4d}  "
        f"Y_dense={Yd:.15f}  "
        f"Y_bessel_upper={rb['Y']:.15f}  "
        f"absdiff={diff:.3e}  "
        f"rel={rel:.3e}  "
        f"time={time.time()-t0:.2f}s"
    )

print()

# ------------------------------------------------------------
# Exact reduction display
# ------------------------------------------------------------

def ceiling_from_B(B):
    """
    If Y_L <= A_LOG*x + B for all x>=0, then this is the maximum of
        T_C <= 36(Ax+B)/(BETA0+GAMMA*x)^2.
    """
    x_star = BETA0 / GAMMA - 2.0 * B / A_LOG
    if x_star < 0:
        x_star = 0.0
    beta_star = BETA0 + GAMMA * x_star
    Tbar = 36.0 * (A_LOG * x_star + B) / (beta_star * beta_star)
    return x_star, beta_star, Tbar

print("=" * 100)
print("CEILING REDUCTION")
print("=" * 100)

for label, B in [
    ("finite max B_L", max_B_row["B_L"]),
    ("safe B = 0.020", 0.020),
    ("safe B = 0.025", 0.025),
    ("safe B = 0.030", 0.030),
    ("safe B = 0.040", 0.040),
    ("critical Bcrit", B_CRIT),
]:
    x_star, beta_star, Tbar = ceiling_from_B(B)
    print(
        f"{label:16s}  "
        f"B={B:.12f}  "
        f"x*={x_star:.6f}  "
        f"beta*={beta_star:.6f}  "
        f"Tbar={Tbar:.12f}  "
        f"floor(1/Tbar)={int(1.0 // Tbar):3d}"
    )

print()
print("M5 PIN STATEMENT")
print("-" * 100)
print("Remaining analytic lemma:")
print()
print("    For all integer L >= 4,")
print("    alpha_W(L)^2 T_C(L) - (3/(32*pi^2))*log(L/4) <= 57/1210.")
print()
print("If proved, then:")
print()
print("    T_C(L) < 1/8")
print("    N*_C(L) >= 8")
print("    T_full(L) < 1/64 + 1/8 = 9/64")
print("    N*(L) >= 7")
print()
print("This script supplies the finite heat-kernel certificate up to L_SCAN_MAX.")
print("The only remaining non-computational object is the theta-tail lemma beyond the scan.")
print("=" * 100)

M5.1 HEAT-KERNEL/BESSEL FINITE CERTIFICATE
L_SCAN_MAX        = 4096
gamma             = 0.139316627508214441
A_log             = 0.009498860966469166
B_crit = 57/1210  = 0.047107438016528926
threshold T_C     = 1/8 = 0.125000000000000000

Scanning integer L...
L=    4  Y=0.016409733262  B_L=0.016409733262  margin=0.030697704754  T_C=0.018837704000  N*_C= 53
L=    5  Y=0.018929513065  B_L=0.016809903495  margin=0.030297534521  T_C=0.021491037164  N*_C= 46
L=    6  Y=0.020959788397  B_L=0.017108331708  margin=0.029999106308  T_C=0.023582815295  N*_C= 42
L=    8  Y=0.024072902796  B_L=0.017488794098  margin=0.029618643918  T_C=0.026705736195  N*_C= 37
L=   12  Y=0.028277464892  B_L=0.017841899506  margin=0.029265538510  T_C=0.030757138402  N*_C= 32
L=   16  Y=0.031164850936  B_L=0.017996633541  margin=0.029110804475  T_C=0.033430311864  N*_C= 29
L=   32  Y=0.037943192191  B_L=0.018190866098  margin=0.028916571918  T_C=0.039377659840  N*_C= 25
L=   64  Y=0.044609778729  B_L=0.018273343939 